# Path-Integrated TorqueTangent v2 (A100)

The first TorqueTangent pilot produced a useful methodological null: behavior changed, but one center gradient did not reconstruct the finite endpoint change, and halving the displacement changed the hidden tangent direction.

[实验设置与运行说明](../../../docs/experiment_notes/12_cross_study_audits_and_pilots/1025bb1d18751002.md)


In [ ]:
# Run this helper once. Every later code cell uses %%cell_progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(
        total=total, initial=initial, desc=label, unit='cell',
        dynamic_ncols=True, leave=True,
    )
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 16 tracked cells.')


## 1. Setup

In Colab, add a read token named `HF_TOKEN` under Secrets if Hugging Face access is rate-limited.


In [ ]:
%%cell_progress 1/16 Mount Drive and locate VisualTorque
cell_stage('Mounting Google Drive')
from google.colab import drive
drive.mount('/content/drive')

cell_stage('Locating VisualTorque project')
from pathlib import Path
PROJECT_CANDIDATES = (
    Path('/content/drive/MyDrive/VisualTorque'),
    Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),
)
PROJECT_ROOT = next(
    (
        path for path in PROJECT_CANDIDATES
        if (path / 'toolkit' / 'pyproject.toml').is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, (
    'Could not find VisualTorque/toolkit/pyproject.toml on Drive.'
)
TOOLKIT_ROOT = PROJECT_ROOT / 'toolkit'
print('Project:', PROJECT_ROOT)


In [ ]:
%%cell_progress 2/16 Install and verify the multimodal environment
import importlib
import subprocess
import sys

cell_stage('Removing unused TorchAudio')
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'],
    check=False,
)
cell_stage('Installing pinned inference and analysis dependencies')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers==5.14.0', 'accelerate>=1.10', 'sentencepiece',
    'protobuf', 'huggingface_hub>=0.34', 'tqdm>=4.66',
    'scikit-learn>=1.5', 'scipy>=1.13', 'pandas>=2.2',
    'matplotlib>=3.8', 'pillow>=10.0',
])
cell_stage('Installing VisualTorque toolkit')
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', str(TOOLKIT_ROOT)
])
SRC_ROOT = TOOLKIT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
importlib.invalidate_caches()

import torch
import torchvision
import visual_torque_toolkit
from transformers import AutoModelForMultimodalLM, AutoProcessor
from visual_torque_toolkit.behavior.adapter import HuggingFaceMultimodalAdapter
required_features = (
    '_template', '_to_device', '_candidate_plan', '_append_text_prefix',
)
assert all(
    hasattr(HuggingFaceMultimodalAdapter, name) for name in required_features
)
print('torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('torchvision:', torchvision.__version__)
print('Toolkit:', visual_torque_toolkit.__file__)


In [ ]:
%%cell_progress 3/16 Verify A100 and Hugging Face access
cell_stage('Checking CUDA device')
assert torch.cuda.is_available(), 'Select an A100 GPU runtime.'
device_name = torch.cuda.get_device_name(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
print(device_name, '| CUDA memory (GB):', round(total_gb, 1))
assert total_gb >= 38, 'This notebook is configured for an A100 40 GB or larger.'

cell_stage('Authenticating with Hugging Face')
from google.colab import userdata
from huggingface_hub import login
try:
    token = userdata.get('HF_TOKEN')
    if token:
        login(token=token, add_to_git_credential=False)
        print('Hugging Face login complete.')
    else:
        print('HF_TOKEN is not set. Add it under Colab Secrets if access is denied.')
except Exception as exc:
    print('Hugging Face login skipped:', exc)


In [ ]:
%%cell_progress 4/16 Lock the path-integral pilot before selecting scenes
cell_stage('Writing the path-integral protocol')
import hashlib
import json
from datetime import datetime, timezone

MODEL_ID = 'Qwen/Qwen3.5-9B'
# Do not enable Gemma until this 12-scene Qwen estimator passes every gate.
# MODEL_ID = 'google/gemma-4-12B-it'

MODEL_SPECS = {
    'Qwen/Qwen3.5-9B': {
        'slug': 'Qwen__Qwen3.5-9B', 'expected_layer': 13,
        'expected_alpha': 0.01,
    },
    'google/gemma-4-12B-it': {
        'slug': 'google__gemma-4-12B-it', 'expected_layer': 7,
        'expected_alpha': 100.0,
    },
}
MODEL_DTYPE = 'bfloat16'
DEVICE_MAP = 'auto'
PILOT_SCENES = 12
MAX_TORQUE_PIXEL_OFFSET = 4.0
FINE_PATH_POINTS = 17
COARSE_PATH_POINTS = 9
BOOTSTRAP_REPLICATES = 5000
PERMUTATION_REPLICATES = 5000
SEED = 20260822
OPTION_VARIANT = 'permutation_0'
assert FINE_PATH_POINTS == 2 * COARSE_PATH_POINTS - 1

spec = MODEL_SPECS[MODEL_ID]
MODEL_ROOT = PROJECT_ROOT / 'results' / spec['slug']
DISCOVERY_ROOT = MODEL_ROOT / 'activation_discovery_v1'
OUTPUT_ROOT = MODEL_ROOT / 'torque_tangent_path_integral_pilot_v2'
SHARD_DIR = OUTPUT_ROOT / 'shards'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SHARD_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
PROTOCOL_LOCK_PATH = OUTPUT_ROOT / 'protocol_lock.json'

protocol_identity = {
    'schema_version': 'path-integrated-torque-tangent-pilot-2.0',
    'analysis_status': 'estimator_repair_after_center_gradient_failure',
    'model_id': MODEL_ID,
    'dtype': MODEL_DTYPE,
    'expected_layer': spec['expected_layer'],
    'expected_alpha': spec['expected_alpha'],
    'center_split': 'probe_test',
    'prompt_condition': 'image_rule',
    'question_kind': 'outcome',
    'option_variant': OPTION_VARIANT,
    'pilot_scenes': PILOT_SCENES,
    'path_pixel_range': [
        -MAX_TORQUE_PIXEL_OFFSET, MAX_TORQUE_PIXEL_OFFSET,
    ],
    'fine_path_points': FINE_PATH_POINTS,
    'coarse_path_points': COARSE_PATH_POINTS,
    'renderer': (
        'static RGB background plus target RGBA stack layer translated with '
        'bilinear subpixel resampling; zero offset must equal Dataset v1 exactly'
    ),
    'signed_torque': 'left_total_minus_right_total',
    'path_orientation': 'positive path coordinate always increases signed torque',
    'margin': 'next_token_semantic_left_logit_minus_right_logit',
    'gradient_site': 'all tokens at frozen selected decoder block output',
    'integration': 'trapezoidal line integral over natural hidden-state path',
    'mediation_site': 'frozen-probe component of final prompt token path increments',
    'no_activation_value_edits': True,
    'bootstrap_replicates': BOOTSTRAP_REPLICATES,
    'permutation_replicates': PERMUTATION_REPLICATES,
    'seed': SEED,
}

def write_json_atomic(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(path)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return 'sha256:' + digest.hexdigest()

if PROTOCOL_LOCK_PATH.is_file():
    previous = json.loads(PROTOCOL_LOCK_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == protocol_identity, (
        'Existing path-integral protocol differs. Use a new versioned output directory.'
    )
else:
    write_json_atomic(PROTOCOL_LOCK_PATH, {
        'identity': protocol_identity,
        'locked_at_utc': datetime.now(timezone.utc).isoformat(),
    })
print('Model:', MODEL_ID)
print('Protocol:', PROTOCOL_LOCK_PATH)
print('Paths/points:', PILOT_SCENES, '/', FINE_PATH_POINTS)


In [ ]:
%%cell_progress 5/16 Prepare Dataset v1 on local SSD
cell_stage('Checking local and Drive dataset caches')
from zipfile import ZipFile

REQUIRED_DATASET_FILES = {
    'dataset_config.json', 'scenes.jsonl', 'counterfactual_pairs.jsonl',
    'questions.jsonl', 'splits.json', 'statistics.json',
}
DATASET_DIR = Path('/content/visualtorque_dataset_v1')
DATASET_ARCHIVE = PROJECT_ROOT / 'data' / 'dataset_v1.zip'
LEGACY_DATASET_DIR = PROJECT_ROOT / 'data' / 'dataset_v1'

def dataset_complete(path):
    return path.is_dir() and all(
        (path / name).is_file() for name in REQUIRED_DATASET_FILES
    )

if not dataset_complete(DATASET_DIR):
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    if DATASET_ARCHIVE.is_file():
        cell_stage('Extracting Dataset v1 archive')
        with ZipFile(DATASET_ARCHIVE, 'r') as bundle:
            members = [m for m in bundle.infolist() if not m.is_dir()]
            root = DATASET_DIR.resolve()
            for member in tqdm(members, desc='Extracting Dataset v1', unit='file'):
                target = (DATASET_DIR / member.filename).resolve()
                if root not in target.parents:
                    raise ValueError(f'Unsafe archive path: {member.filename}')
                bundle.extract(member, DATASET_DIR)
    elif dataset_complete(LEGACY_DATASET_DIR):
        DATASET_DIR = LEGACY_DATASET_DIR
    else:
        raise FileNotFoundError(
            'Run the Dataset v1 build/cache cell in the behavior notebook first.'
        )
assert dataset_complete(DATASET_DIR)
print('Dataset:', DATASET_DIR)


In [ ]:
%%cell_progress 6/16 Load the frozen probe and its held-out training provenance
cell_stage('Loading discovery artifacts')
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def torch_load_trusted(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except TypeError:
        return torch.load(path, map_location='cpu')

discovery_config_path = DISCOVERY_ROOT / 'capture_config.json'
discovery_selection_path = (
    DISCOVERY_ROOT / 'probe_results' / 'discovery_selection.json'
)
discovery_activation_path = DISCOVERY_ROOT / 'scene_mean_activations.pt'
for required in (
    discovery_config_path, discovery_selection_path, discovery_activation_path,
):
    assert required.is_file(), f'Missing prerequisite: {required}'

discovery_config = json.loads(
    discovery_config_path.read_text(encoding='utf-8')
)
discovery_identity = discovery_config['identity']
MODEL_REVISION = str(discovery_identity['resolved_revision'])
selection = json.loads(
    discovery_selection_path.read_text(encoding='utf-8')
)
choice = selection['candidate_net_torque_layer']
SELECTED_LAYER_INDEX = int(choice['layer_index'])
FROZEN_ALPHA = float(choice['selected_alpha'])
assert SELECTED_LAYER_INDEX == spec['expected_layer']
assert abs(FROZEN_ALPHA - spec['expected_alpha']) < 1e-12

discovery = torch_load_trusted(discovery_activation_path)
layer_indices = [int(value) for value in discovery['layer_indices']]
layer_position = layer_indices.index(SELECTED_LAYER_INDEX)
X_fit = discovery['activations'][:, layer_position, :].float().numpy()
fit_metadata = pd.DataFrame(discovery['metadata'])
assert set(fit_metadata['split']) <= {'probe_train', 'probe_dev'}
y_fit = pd.to_numeric(
    fit_metadata['net_torque'], errors='raise'
).to_numpy(dtype=float)

probe_scaler = StandardScaler()
probe = Ridge(alpha=FROZEN_ALPHA, solver='lsqr', max_iter=5000)
probe.fit(probe_scaler.fit_transform(X_fit), y_fit)
PROBE_MEAN = probe_scaler.mean_.astype(np.float32)
PROBE_SCALE = probe_scaler.scale_.astype(np.float32)
PROBE_COEF = np.asarray(probe.coef_, dtype=np.float32)
PROBE_INTERCEPT = float(probe.intercept_)
PROBE_COEF_NORM_SQUARED = float(np.dot(PROBE_COEF, PROBE_COEF))
assert PROBE_COEF_NORM_SQUARED > 0
# If q_z is a tangent in standardized coordinates, its projection is
# coef * (q_z dot coef)/(coef dot coef). Mapping back gives this raw-h basis.
PROBE_DIRECTION_H = (PROBE_SCALE * PROBE_COEF).astype(np.float32)

probe_summary = {
    'selected_layer_index': SELECTED_LAYER_INDEX,
    'frozen_alpha': FROZEN_ALPHA,
    'fit_scenes': len(y_fit),
    'fit_splits': sorted(set(fit_metadata['split'])),
    'hidden_size': int(X_fit.shape[1]),
    'probe_coefficient_norm': float(np.linalg.norm(PROBE_COEF)),
    'model_revision': MODEL_REVISION,
}
print(json.dumps(probe_summary, indent=2))


## 2. Construct smooth, held-out, pixel-local torque paths

Each center is a unique `probe_test` single-stack scene.


In [ ]:
%%cell_progress 7/16 Select centers and render 17-point subpixel paths
cell_stage('Loading image-rule outcome prompts')
from collections import Counter, defaultdict
from dataclasses import replace
from PIL import Image, ImageChops, ImageDraw
import numpy as np

from visual_torque_toolkit.behavior import (
    BehaviorRunConfig, PromptCondition, load_behavior_examples,
)
from visual_torque_toolkit.render import RenderStyle

example_config = BehaviorRunConfig(
    dataset_dir=DATASET_DIR,
    output_dir=Path('/content/unused_path_integral_output'),
    model_id=MODEL_ID,
    model_revision=MODEL_REVISION,
    conditions=(PromptCondition.IMAGE_RULE,),
    question_kinds=('outcome',),
    splits=('probe_train', 'probe_dev', 'probe_test'),
    batch_size=1,
    generation_every=0,
    seed=SEED,
)
all_examples = tuple(load_behavior_examples(example_config))
example_by_scene_variant = {
    (str(example.scene_id), example.variant): example
    for example in all_examples
}
scene_records = read_jsonl(DATASET_DIR / 'scenes.jsonl')
scene_by_id = {str(scene['scene_id']): scene for scene in scene_records}

def stable_key(value):
    return hashlib.sha256(f'{SEED}:{value}'.encode('utf-8')).hexdigest()

def signed_torque(scene):
    left = sum(
        float(stack['units']) * float(stack['distance'])
        for stack in scene['stacks'] if stack['side'] == 'left'
    )
    right = sum(
        float(stack['units']) * float(stack['distance'])
        for stack in scene['stacks'] if stack['side'] == 'right'
    )
    return left - right

candidates = []
for scene in scene_records:
    if scene['split'] != 'probe_test':
        continue
    if scene.get('metadata', {}).get('family') != 'single_stack':
        continue
    if len(scene['stacks']) != 2:
        continue
    for target_side in ('left', 'right'):
        target = next(
            stack for stack in scene['stacks']
            if stack['side'] == target_side
        )
        if not 2 <= int(target['distance']) <= 5:
            continue
        path_id = f"{scene['scene_id']}__vary_{target_side}_subpixel"
        candidates.append({
            'path_id': path_id,
            'center_scene_id': str(scene['scene_id']),
            'center_split': str(scene['split']),
            'target_side': target_side,
            'target_units': int(target['units']),
            'target_center_distance': float(target['distance']),
            'center_signed_torque': float(signed_torque(scene)),
            'center_outcome': str(scene['derived']['outcome']),
            'problem_type': str(scene['problem_type']),
        })

groups = defaultdict(list)
for candidate in candidates:
    groups[(candidate['target_side'], candidate['center_outcome'])].append(candidate)
for values in groups.values():
    values.sort(key=lambda value: stable_key(value['path_id']))
group_keys = sorted(groups)
cursors = {key: 0 for key in group_keys}
selected_paths = []
used_centers = set()
while len(selected_paths) < PILOT_SCENES:
    changed = False
    for key in group_keys:
        values = groups[key]
        while cursors[key] < len(values):
            candidate = values[cursors[key]]
            cursors[key] += 1
            if candidate['center_scene_id'] in used_centers:
                continue
            selected_paths.append(candidate)
            used_centers.add(candidate['center_scene_id'])
            changed = True
            break
        if len(selected_paths) >= PILOT_SCENES:
            break
    if not changed:
        break
assert len(selected_paths) == PILOT_SCENES

STYLE = RenderStyle()
CENTER_X = STYLE.width // 2
HALF_SPAN = int(STYLE.width * 0.38)
PIXELS_PER_TICK = HALF_SPAN / STYLE.ticks_per_side
PATH_COORDINATES = np.linspace(
    -MAX_TORQUE_PIXEL_OFFSET,
    MAX_TORQUE_PIXEL_OFFSET,
    FINE_PATH_POINTS,
    dtype=float,
)
assert np.allclose(PATH_COORDINATES[::2], np.linspace(
    -MAX_TORQUE_PIXEL_OFFSET,
    MAX_TORQUE_PIXEL_OFFSET,
    COARSE_PATH_POINTS,
))
PATH_IMAGE_ROOT = Path('/content/visualtorque_path_integral_images') / spec['slug']
PATH_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)

def draw_static_scene(scene, target_side):
    style = STYLE
    image = Image.new('RGB', (style.width, style.height), style.background)
    draw = ImageDraw.Draw(image)
    beam_left, beam_right = CENTER_X - HALF_SPAN, CENTER_X + HALF_SPAN
    draw.line(
        (beam_left, style.beam_y, beam_right, style.beam_y),
        fill=style.ink, width=7,
    )
    for signed_distance in range(-style.ticks_per_side, style.ticks_per_side + 1):
        x = round(CENTER_X + signed_distance * PIXELS_PER_TICK)
        tick_height = 16 if signed_distance else 22
        draw.line(
            (x, style.beam_y - tick_height // 2,
             x, style.beam_y + tick_height // 2),
            fill=style.ink, width=3,
        )
    fulcrum_top = style.beam_y + 8
    fulcrum_bottom = style.beam_y + 104
    draw.polygon(
        ((CENTER_X, fulcrum_top), (CENTER_X - 66, fulcrum_bottom),
         (CENTER_X + 66, fulcrum_bottom)),
        fill=(214, 171, 79), outline=style.ink,
    )
    for stack in scene['stacks']:
        if stack['side'] == target_side:
            continue
        sign = -1 if stack['side'] == 'left' else 1
        x_center = round(
            CENTER_X + sign * float(stack['distance']) * PIXELS_PER_TICK
        )
        for level in range(int(stack['units'])):
            bottom = style.beam_y - 7 - level * (
                style.block_size + style.block_gap
            )
            top = bottom - style.block_size
            left = x_center - style.block_size // 2
            right = left + style.block_size
            draw.rectangle(
                (left, top, right, bottom), fill=style.block_fill,
                outline=style.ink, width=2,
            )
    return image

def draw_target_layer(scene, target_side):
    style = STYLE
    layer = Image.new('RGBA', (style.width, style.height), (0, 0, 0, 0))
    draw = ImageDraw.Draw(layer)
    target = next(
        stack for stack in scene['stacks'] if stack['side'] == target_side
    )
    sign = -1 if target_side == 'left' else 1
    x_center = round(
        CENTER_X + sign * float(target['distance']) * PIXELS_PER_TICK
    )
    for level in range(int(target['units'])):
        bottom = style.beam_y - 7 - level * (
            style.block_size + style.block_gap
        )
        top = bottom - style.block_size
        left = x_center - style.block_size // 2
        right = left + style.block_size
        draw.rectangle(
            (left, top, right, bottom),
            fill=tuple(style.block_fill) + (255,),
            outline=tuple(style.ink) + (255,), width=2,
        )
    return layer

def translate_layer(layer, screen_dx):
    if abs(float(screen_dx)) < 1e-12:
        return layer
    affine = getattr(Image, 'Transform', Image).AFFINE
    resampling = getattr(Image, 'Resampling', Image).BILINEAR
    return layer.transform(
        layer.size, affine,
        (1.0, 0.0, -float(screen_dx), 0.0, 1.0, 0.0),
        resample=resampling, fillcolor=(0, 0, 0, 0),
    )

def render_path_point(scene, target_side, torque_pixel_coordinate, output_path):
    static = draw_static_scene(scene, target_side)
    target = draw_target_layer(scene, target_side)
    # For either side, moving the target left increases left-minus-right torque.
    screen_dx = -float(torque_pixel_coordinate)
    shifted = translate_layer(target, screen_dx)
    rendered = Image.alpha_composite(static.convert('RGBA'), shifted).convert('RGB')
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    rendered.save(output_path, format='PNG')
    return output_path.resolve()

cell_stage('Rendering smooth local paths on the Colab SSD')
rendered_paths = []
center_audits = []
background_audits = []
for path in tqdm(selected_paths, desc='Rendering 17-point paths', unit='path'):
    scene = scene_by_id[path['center_scene_id']]
    path_dir = PATH_IMAGE_ROOT / path['path_id']
    points = []
    for point_index, coordinate in enumerate(PATH_COORDINATES):
        image_path = render_path_point(
            scene, path['target_side'], coordinate,
            path_dir / f'point_{point_index:02d}.png',
        )
        points.append({
            'point_index': point_index,
            'torque_pixel_coordinate': float(coordinate),
            'screen_dx': float(-coordinate),
            'signed_torque': float(
                path['center_signed_torque']
                + path['target_units'] * coordinate / PIXELS_PER_TICK
            ),
            'image_path': str(image_path),
        })
    center_point = points[FINE_PATH_POINTS // 2]
    original_path = (DATASET_DIR / scene['image_path']).resolve()
    with Image.open(center_point['image_path']) as rendered, Image.open(
        original_path
    ) as original:
        center_exact = ImageChops.difference(
            rendered.convert('RGB'), original.convert('RGB')
        ).getbbox() is None
    center_audits.append(center_exact)
    assert center_exact, f'Center renderer mismatch: {path["path_id"]}'

    # Differences must be confined to the target stack's translated envelope.
    with Image.open(points[0]['image_path']) as low, Image.open(
        points[-1]['image_path']
    ) as high:
        difference_box = ImageChops.difference(
            low.convert('RGB'), high.convert('RGB')
        ).getbbox()
    background_audits.append(difference_box is not None)
    assert difference_box is not None
    rendered = dict(path)
    rendered['pixels_per_tick'] = float(PIXELS_PER_TICK)
    rendered['points'] = points
    rendered_paths.append(rendered)

path_manifest_path = OUTPUT_ROOT / 'path_manifest.json'
manifest_paths = []
for path in rendered_paths:
    manifest_paths.append({
        **{key: value for key, value in path.items() if key != 'points'},
        'points': [
            {key: value for key, value in point.items() if key != 'image_path'}
            for point in path['points']
        ],
    })
write_json_atomic(path_manifest_path, {
    'schema_version': 'path-integral-manifest-2.0',
    'dataset_fingerprint': example_config.dataset_fingerprint(),
    'paths': manifest_paths,
})

cell_stage('Hashing frozen artifacts for append-only resume')
capture_identity = {
    'schema_version': 'path-integral-capture-2.0',
    'protocol_lock_sha256': sha256_file(PROTOCOL_LOCK_PATH),
    'dataset_fingerprint': example_config.dataset_fingerprint(),
    'discovery_config_sha256': sha256_file(discovery_config_path),
    'discovery_selection_sha256': sha256_file(discovery_selection_path),
    'discovery_activations_sha256': sha256_file(discovery_activation_path),
    'path_manifest_sha256': sha256_file(path_manifest_path),
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'selected_layer_index': SELECTED_LAYER_INDEX,
    'frozen_alpha': FROZEN_ALPHA,
    'expected_paths': len(rendered_paths),
}
CAPTURE_CONFIG_PATH = OUTPUT_ROOT / 'capture_config.json'
if CAPTURE_CONFIG_PATH.is_file():
    previous = json.loads(CAPTURE_CONFIG_PATH.read_text(encoding='utf-8'))
    assert previous.get('identity') == capture_identity, (
        'Existing capture uses different path/probe artifacts. '
        'Use a new versioned output directory.'
    )
else:
    write_json_atomic(CAPTURE_CONFIG_PATH, {
        'identity': capture_identity,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
    })

def example_for_point(path, point):
    base = example_by_scene_variant[(
        path['center_scene_id'], OPTION_VARIANT,
    )]
    return replace(
        base,
        example_id=f"path_integral:{path['path_id']}:{point['point_index']}",
        scene_id=f"{path['path_id']}:{point['point_index']}",
        image_path=Path(point['image_path']),
        net_torque=point['signed_torque'],
    )

print('Selected paths:', len(rendered_paths))
print('Strata:', Counter(
    (path['target_side'], path['center_outcome']) for path in rendered_paths
))
print('Path coordinates:', PATH_COORDINATES.tolist())
print('Center pixel audits:', sum(center_audits), '/', len(center_audits))
print('Pixels per tick:', round(PIXELS_PER_TICK, 4))


In [ ]:
%%cell_progress 8/16 Load the pinned model and freeze its parameters
cell_stage('Loading processor and pinned model weights')
adapter = HuggingFaceMultimodalAdapter(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=MODEL_DTYPE,
    device_map=DEVICE_MAP,
)
display(adapter.metadata)

cell_stage('Resolving decoder layers')
def resolve_module_path(root, path):
    value = root
    for component in path.split('.'):
        value = getattr(value, component)
    return value

def discover_decoder_layers(model):
    candidates = [
        discovery_identity.get('layer_path'),
        'model.language_model.layers', 'model.text_model.layers',
        'language_model.model.layers', 'language_model.layers',
        'model.model.layers', 'model.layers',
    ]
    failures = []
    for path in candidates:
        if not path:
            continue
        try:
            layers = resolve_module_path(model, path)
        except (AttributeError, KeyError) as exc:
            failures.append(f'{path}: {exc}')
            continue
        if isinstance(layers, torch.nn.ModuleList) and len(layers) >= 8:
            return path, layers
        failures.append(f'{path}: {type(layers).__name__}')
    raise RuntimeError('Could not locate decoder layers:\n' + '\n'.join(failures))

LAYER_PATH, DECODER_LAYERS = discover_decoder_layers(adapter.model)
assert len(DECODER_LAYERS) == int(discovery_identity['decoder_layer_count'])
TARGET_LAYER = DECODER_LAYERS[SELECTED_LAYER_INDEX]
resolved_now = adapter.metadata.get('resolved_revision')
if resolved_now is not None:
    assert str(resolved_now) == MODEL_REVISION

for parameter in adapter.model.parameters():
    parameter.requires_grad_(False)
adapter.model.eval()
torch.cuda.empty_cache()
print('Layer path:', LAYER_PATH)
print('Decoder layers:', len(DECODER_LAYERS))
print('Target layer:', SELECTED_LAYER_INDEX)
print('Trainable model parameters:', sum(
    parameter.requires_grad for parameter in adapter.model.parameters()
))


## 3. Natural-path gradients and segment-wise line integration

The hook substitutes an equal-valued detached leaf only to cut backward memory below the selected layer.


In [ ]:
%%cell_progress 9/16 Define exact point gradients and path integration
cell_stage('Defining safe hidden capture and cut-gradient functions')
import gc
import time

def output_hidden_tensor(output):
    if torch.is_tensor(output):
        hidden = output
    elif isinstance(output, (tuple, list)) and output and torch.is_tensor(output[0]):
        hidden = output[0]
    elif hasattr(output, 'last_hidden_state') and torch.is_tensor(
        output.last_hidden_state
    ):
        hidden = output.last_hidden_state
    else:
        raise TypeError(f'Unsupported decoder output: {type(output).__name__}')
    if hidden.ndim != 3:
        raise ValueError(f'Expected [batch, sequence, hidden], got {hidden.shape}')
    return hidden

def replace_output_hidden(output, modified):
    if torch.is_tensor(output):
        return modified
    if isinstance(output, tuple):
        return (modified,) + output[1:]
    if isinstance(output, list):
        return [modified] + output[1:]
    raise TypeError(
        'Decoder output object is not safely replaceable: '
        + type(output).__name__
    )

def prepared_inputs_and_tokens(example):
    plan = adapter._candidate_plan(example)
    if plan is None:
        raise RuntimeError(
            'Path integration requires single-token A/B/C labels.'
        )
    fixed_prefix, token_by_label = plan
    inputs = adapter._append_text_prefix(
        adapter._to_device(
            adapter._template([example.messages()], add_generation_prompt=True)
        ),
        fixed_prefix,
    )
    semantic_to_label = {
        str(semantic): label
        for label, semantic in example.label_to_semantic.items()
    }
    token_ids = {
        semantic: int(token_by_label[semantic_to_label[semantic]])
        for semantic in ('left', 'right', 'balanced')
    }
    return inputs, token_ids

def semantic_margin(logits, token_ids):
    return logits[0, -1, token_ids['left']].float() - logits[
        0, -1, token_ids['right']
    ].float()

def no_cut_margin(example):
    inputs, token_ids = prepared_inputs_and_tokens(example)
    with torch.inference_mode():
        output = adapter.model(**inputs, use_cache=False)
        margin = semantic_margin(output.logits, token_ids)
    value = float(margin.detach().cpu())
    del inputs, output, margin
    return value

def point_cut_gradient(example):
    captured = {}

    def cut_hook(_module, _inputs, output):
        hidden = output_hidden_tensor(output)
        leaf = hidden.detach().requires_grad_(True)
        captured['leaf'] = leaf
        return replace_output_hidden(output, leaf)

    inputs, token_ids = prepared_inputs_and_tokens(example)
    handle = TARGET_LAYER.register_forward_hook(cut_hook)
    started = time.perf_counter()
    try:
        with torch.enable_grad():
            output = adapter.model(**inputs, use_cache=False)
            margin = semantic_margin(output.logits, token_ids)
            gradient = torch.autograd.grad(
                margin, captured['leaf'], retain_graph=False,
                create_graph=False, allow_unused=False,
            )[0]
    finally:
        handle.remove()
    result = {
        'hidden': captured['leaf'][0].detach().float().cpu(),
        'gradient': gradient[0].detach().float().cpu(),
        'margin': float(margin.detach().cpu()),
        'latency_seconds': time.perf_counter() - started,
    }
    del inputs, output, margin, gradient, captured
    return result

probe_mean_t = torch.from_numpy(PROBE_MEAN).float()
probe_scale_t = torch.from_numpy(PROBE_SCALE).float()
probe_coef_t = torch.from_numpy(PROBE_COEF).float()
probe_direction_h_t = torch.from_numpy(PROBE_DIRECTION_H).float()

def decoded_torque(hidden_last):
    standardized = (hidden_last.float() - probe_mean_t) / probe_scale_t
    return float(standardized @ probe_coef_t + PROBE_INTERCEPT)

def integrate_point_results(point_results, point_records, stride):
    indices = list(range(0, len(point_results), stride))
    assert indices[-1] == len(point_results) - 1
    segment_rows = []
    for left_index, right_index in zip(indices[:-1], indices[1:]):
        left, right = point_results[left_index], point_results[right_index]
        gradient_mean = 0.5 * (left['gradient'] + right['gradient'])
        delta_hidden = right['hidden'] - left['hidden']
        token_contribution = (gradient_mean * delta_hidden).sum(dim=1)
        full = float(token_contribution.sum())
        final_full = float(token_contribution[-1])

        delta_z_last = delta_hidden[-1] / probe_scale_t
        coordinate = float(
            (delta_z_last @ probe_coef_t) / PROBE_COEF_NORM_SQUARED
        )
        sensitivity = float(gradient_mean[-1] @ probe_direction_h_t)
        probe_contribution = coordinate * sensitivity
        observed = float(right['margin'] - left['margin'])
        segment_rows.append({
            'left_point_index': int(left_index),
            'right_point_index': int(right_index),
            'left_path_coordinate': float(
                point_records[left_index]['torque_pixel_coordinate']
            ),
            'right_path_coordinate': float(
                point_records[right_index]['torque_pixel_coordinate']
            ),
            'observed_margin_change': observed,
            'full_integral_contribution': full,
            'final_token_full_contribution': final_full,
            'probe_contribution': float(probe_contribution),
            'final_token_orthogonal_contribution': float(
                final_full - probe_contribution
            ),
            'other_token_contribution': float(full - final_full),
            'probe_coordinate': coordinate,
            'probe_sensitivity': sensitivity,
            'decoded_torque_change': float(delta_z_last @ probe_coef_t),
        })
    first, last = point_results[indices[0]], point_results[indices[-1]]
    return {
        'point_stride': int(stride),
        'point_count': len(indices),
        'segment_count': len(segment_rows),
        'observed_margin_change': float(last['margin'] - first['margin']),
        'full_integral': float(sum(
            row['full_integral_contribution'] for row in segment_rows
        )),
        'final_token_full_integral': float(sum(
            row['final_token_full_contribution'] for row in segment_rows
        )),
        'probe_integral': float(sum(
            row['probe_contribution'] for row in segment_rows
        )),
        'final_token_orthogonal_integral': float(sum(
            row['final_token_orthogonal_contribution'] for row in segment_rows
        )),
        'other_token_integral': float(sum(
            row['other_token_contribution'] for row in segment_rows
        )),
        'decoded_torque_change': float(sum(
            row['decoded_torque_change'] for row in segment_rows
        )),
        'actual_torque_change': float(
            point_records[indices[-1]]['signed_torque']
            - point_records[indices[0]]['signed_torque']
        ),
        'segments': segment_rows,
    }

def evaluate_path(path, show_point_progress=False):
    point_results = []
    iterator = path['points']
    if show_point_progress:
        iterator = tqdm(
            iterator, desc='Path points', unit='point', leave=False,
            dynamic_ncols=True,
        )
    for point in iterator:
        point_results.append(point_cut_gradient(
            example_for_point(path, point)
        ))
    shapes = {tuple(result['hidden'].shape) for result in point_results}
    assert len(shapes) == 1
    fine = integrate_point_results(point_results, path['points'], stride=1)
    coarse = integrate_point_results(point_results, path['points'], stride=2)
    center_result = point_results[len(point_results) // 2]
    record = {
        'schema_version': 'path-integrated-torque-record-2.0',
        'path_id': path['path_id'],
        'center_scene_id': path['center_scene_id'],
        'center_split': path['center_split'],
        'target_side': path['target_side'],
        'target_units': int(path['target_units']),
        'target_center_distance': float(path['target_center_distance']),
        'center_signed_torque': float(path['center_signed_torque']),
        'center_outcome': path['center_outcome'],
        'problem_type': path['problem_type'],
        'selected_layer_index': SELECTED_LAYER_INDEX,
        'sequence_tokens': int(center_result['hidden'].shape[0]),
        'center_margin': float(center_result['margin']),
        'center_decoded_torque': decoded_torque(
            center_result['hidden'][-1]
        ),
        'total_gradient_latency_seconds': float(sum(
            result['latency_seconds'] for result in point_results
        )),
        'fine': fine,
        'coarse': coarse,
        'coarse_fine_full_difference': float(
            coarse['full_integral'] - fine['full_integral']
        ),
        'point_margins': [
            float(result['margin']) for result in point_results
        ],
        'point_decoded_torques': [
            decoded_torque(result['hidden'][-1]) for result in point_results
        ],
    }
    del point_results
    gc.collect()
    return record

print('Path-gradient and integration functions ready.')


In [ ]:
%%cell_progress 10/16 Run the mandatory one-path estimator smoke test
cell_stage('Checking equal-valued cut-gradient parity at the center')
protocol_hash = sha256_file(PROTOCOL_LOCK_PATH)
capture_hash = sha256_file(CAPTURE_CONFIG_PATH)
smoke_path = rendered_paths[0]
center_point = smoke_path['points'][FINE_PATH_POINTS // 2]
center_example = example_for_point(smoke_path, center_point)
reference_margin = no_cut_margin(center_example)
cut_center = point_cut_gradient(center_example)
parity_error = abs(reference_margin - cut_center['margin'])
assert parity_error < 1e-4, ('Cut hook changed the forward value', parity_error)
del cut_center

smoke_shard_path = SHARD_DIR / f"{smoke_path['path_id']}.json"
if smoke_shard_path.is_file():
    smoke_record = json.loads(smoke_shard_path.read_text(encoding='utf-8'))
    assert smoke_record['protocol_lock_sha256'] == protocol_hash
    assert smoke_record['capture_config_sha256'] == capture_hash
    print('Resuming the existing smoke-path shard.')
else:
    smoke_record = evaluate_path(smoke_path, show_point_progress=True)
    smoke_record['center_cut_parity_error'] = float(parity_error)
    smoke_record['protocol_lock_sha256'] = protocol_hash
    smoke_record['capture_config_sha256'] = capture_hash
    smoke_record['created_at_utc'] = datetime.now(timezone.utc).isoformat()
    write_json_atomic(smoke_shard_path, smoke_record)
observed = smoke_record['fine']['observed_margin_change']
fine_full = smoke_record['fine']['full_integral']
coarse_full = smoke_record['coarse']['full_integral']
scale = max(abs(observed), 1e-6)
print('Smoke path:', smoke_record['path_id'])
print('Observed endpoint margin change:', round(observed, 6))
print('17-point full integral:', round(fine_full, 6))
print('9-point full integral:', round(coarse_full, 6))
print('17-point normalized error:', round(abs(fine_full - observed) / scale, 4))
print('9-vs-17 normalized difference:', round(
    abs(coarse_full - fine_full) / max(abs(fine_full), scale, 1e-6), 4
))
print('Probe integral (diagnostic until all gates pass):', round(
    smoke_record['fine']['probe_integral'], 8
))
print('Cut parity error:', parity_error)
torch.cuda.empty_cache()


In [ ]:
%%cell_progress 11/16 Capture or resume all 12 natural paths
cell_stage('Reading completed path-integral shards')
expected_path_ids = {path['path_id'] for path in rendered_paths}
completed_path_ids = set()
for shard_path in SHARD_DIR.glob('*.json'):
    payload = json.loads(shard_path.read_text(encoding='utf-8'))
    if payload.get('path_id') not in expected_path_ids:
        continue
    assert payload.get('protocol_lock_sha256') == protocol_hash
    assert payload.get('capture_config_sha256') == capture_hash
    if payload['path_id'] in completed_path_ids:
        raise RuntimeError(f"Duplicate path shard: {payload['path_id']}")
    completed_path_ids.add(payload['path_id'])
remaining = [
    path for path in rendered_paths if path['path_id'] not in completed_path_ids
]
print('Completed:', len(completed_path_ids), '| remaining:', len(remaining))

cell_stage('Running 17 unmodified point gradients per remaining path')
started = time.perf_counter()
for path_index, path in enumerate(tqdm(
    remaining, desc='Integrated torque paths', unit='path', dynamic_ncols=True
), start=1):
    record = evaluate_path(path, show_point_progress=True)
    record['center_cut_parity_error'] = None
    record['protocol_lock_sha256'] = protocol_hash
    record['capture_config_sha256'] = capture_hash
    record['created_at_utc'] = datetime.now(timezone.utc).isoformat()
    write_json_atomic(SHARD_DIR / f"{path['path_id']}.json", record)
    gc.collect()
    torch.cuda.empty_cache()
    elapsed = time.perf_counter() - started
    rate = elapsed / path_index
    eta = rate * (len(remaining) - path_index)
    print(
        f'Checkpoint {len(completed_path_ids)+path_index}/{PILOT_SCENES} | '
        f'elapsed {elapsed/60:.1f} min | ETA {eta/60:.1f} min'
    )

cell_stage('Verifying complete append-only capture')
final_ids = set()
for path_id in expected_path_ids:
    shard_path = SHARD_DIR / f'{path_id}.json'
    assert shard_path.is_file(), f'Missing shard: {path_id}'
    payload = json.loads(shard_path.read_text(encoding='utf-8'))
    assert payload['protocol_lock_sha256'] == protocol_hash
    assert payload['capture_config_sha256'] == capture_hash
    final_ids.add(payload['path_id'])
assert final_ids == expected_path_ids
print('All path-integral pilots complete:', len(final_ids))


## 4. Validate the estimator before interpreting the probe

The primary gate compares the 17-point all-token integral with the observed endpoint answer change.


In [ ]:
%%cell_progress 12/16 Aggregate path and segment records
cell_stage('Loading complete path-integral records')
import pandas as pd

records = []
for path_id in sorted(expected_path_ids):
    record = json.loads(
        (SHARD_DIR / f'{path_id}.json').read_text(encoding='utf-8')
    )
    assert record['protocol_lock_sha256'] == protocol_hash
    assert record['capture_config_sha256'] == capture_hash
    records.append(record)
assert len(records) == PILOT_SCENES

path_rows = []
fine_segment_rows = []
for record in records:
    fine, coarse = record['fine'], record['coarse']
    row = {
        key: record[key] for key in (
            'path_id', 'center_scene_id', 'center_split', 'target_side',
            'target_units', 'target_center_distance', 'center_signed_torque',
            'center_outcome', 'problem_type', 'selected_layer_index',
            'sequence_tokens', 'center_margin', 'center_decoded_torque',
        )
    }
    row.update({
        'observed_margin_change': fine['observed_margin_change'],
        'fine_full_integral': fine['full_integral'],
        'coarse_full_integral': coarse['full_integral'],
        'fine_probe_integral': fine['probe_integral'],
        'fine_final_token_full_integral': fine['final_token_full_integral'],
        'fine_final_token_orthogonal_integral': (
            fine['final_token_orthogonal_integral']
        ),
        'fine_other_token_integral': fine['other_token_integral'],
        'fine_decoded_torque_change': fine['decoded_torque_change'],
        'actual_torque_change': fine['actual_torque_change'],
        'coarse_fine_full_difference': (
            coarse['full_integral'] - fine['full_integral']
        ),
    })
    path_rows.append(row)
    for segment in fine['segments']:
        fine_segment_rows.append({
            'path_id': record['path_id'],
            'target_side': record['target_side'],
            'center_signed_torque': record['center_signed_torque'],
            **segment,
        })
path_frame = pd.DataFrame(path_rows)
segment_frame = pd.DataFrame(fine_segment_rows)
assert len(path_frame) == PILOT_SCENES
assert len(segment_frame) == PILOT_SCENES * (FINE_PATH_POINTS - 1)
assert set(path_frame['center_split']) == {'probe_test'}
assert path_frame.select_dtypes(include=[np.number]).replace(
    [np.inf, -np.inf], np.nan
).notna().all().all()
path_frame.to_csv(ANALYSIS_DIR / 'path_integral_primary.csv', index=False)
segment_frame.to_csv(ANALYSIS_DIR / 'path_integral_segments.csv', index=False)
display(path_frame.head())
print('Path rows:', len(path_frame), '| fine segments:', len(segment_frame))


In [ ]:
%%cell_progress 13/16 Apply reconstruction, convergence, response, and probe gates
cell_stage('Computing path-integral validity diagnostics')
from scipy.stats import pearsonr, spearmanr

observed = path_frame['observed_margin_change'].to_numpy(dtype=float)
fine_full = path_frame['fine_full_integral'].to_numpy(dtype=float)
coarse_full = path_frame['coarse_full_integral'].to_numpy(dtype=float)
denominator = float(np.dot(observed, observed)) + np.finfo(float).eps
reconstruction_slope = float(np.dot(fine_full, observed) / denominator)
reconstruction_pearson = float(pearsonr(fine_full, observed).statistic)
reconstruction_spearman = float(spearmanr(fine_full, observed).statistic)
response_scale = max(float(np.median(np.abs(observed))), 1e-6)
reconstruction_error = np.abs(fine_full - observed) / (
    np.abs(observed) + 0.10 * response_scale
)
reconstruction_median_error = float(np.median(reconstruction_error))

convergence_error = np.abs(coarse_full - fine_full) / (
    np.maximum(np.abs(fine_full), np.abs(observed))
    + 0.10 * response_scale
)
convergence_median_error = float(np.median(convergence_error))
convergence_spearman = float(spearmanr(coarse_full, fine_full).statistic)

segment_observed = segment_frame[
    'observed_margin_change'
].to_numpy(dtype=float)
segment_full = segment_frame[
    'full_integral_contribution'
].to_numpy(dtype=float)
segment_spearman = float(spearmanr(segment_full, segment_observed).statistic)
segment_scale = max(float(np.median(np.abs(segment_observed))), 1e-6)
segment_median_error = float(np.median(
    np.abs(segment_full - segment_observed)
    / (np.abs(segment_observed) + 0.10 * segment_scale)
))

response_rms = float(np.sqrt(np.mean(np.square(observed))))
decoded_gain = (
    path_frame['fine_decoded_torque_change'].to_numpy(dtype=float)
    / path_frame['actual_torque_change'].to_numpy(dtype=float)
)
median_decoded_gain = float(np.median(decoded_gain))
positive_decoded_gain_fraction = float(np.mean(decoded_gain > 0))

reconstruction_gate = bool(
    0.90 <= reconstruction_slope <= 1.10
    and reconstruction_spearman >= 0.90
    and reconstruction_median_error <= 0.15
)
convergence_gate = bool(
    convergence_spearman >= 0.90
    and convergence_median_error <= 0.15
)
response_gate = bool(response_rms >= 0.01)
probe_path_gate = bool(
    median_decoded_gain >= 0.10
    and positive_decoded_gain_fraction >= 0.70
)
path_integral_estimator_valid = bool(
    reconstruction_gate and convergence_gate and response_gate
)
mediation_interpretable = bool(
    path_integral_estimator_valid and probe_path_gate
)

sanity_summary = {
    'schema_version': 'path-integral-sanity-2.0',
    'n_paths': len(path_frame),
    'n_fine_segments': len(segment_frame),
    'reconstruction_17_point': {
        'through_origin_slope': reconstruction_slope,
        'pearson': reconstruction_pearson,
        'spearman': reconstruction_spearman,
        'median_normalized_absolute_error': reconstruction_median_error,
        'gate_passed': reconstruction_gate,
    },
    'convergence_9_to_17': {
        'spearman': convergence_spearman,
        'median_normalized_difference': convergence_median_error,
        'gate_passed': convergence_gate,
    },
    'fine_segment_diagnostics': {
        'spearman': segment_spearman,
        'median_normalized_absolute_error': segment_median_error,
    },
    'behavioral_response': {
        'endpoint_margin_change_rms': response_rms,
        'gate_passed': response_gate,
    },
    'probe_local_path_fidelity': {
        'median_decoded_torque_gain': median_decoded_gain,
        'positive_gain_fraction': positive_decoded_gain_fraction,
        'gate_passed': probe_path_gate,
    },
    'path_integral_estimator_valid': path_integral_estimator_valid,
    'mediation_interpretable': mediation_interpretable,
    'thresholds': {
        'reconstruction_slope_interval': [0.90, 1.10],
        'reconstruction_spearman_min': 0.90,
        'reconstruction_median_error_max': 0.15,
        'convergence_spearman_min': 0.90,
        'convergence_median_difference_max': 0.15,
        'behavioral_response_rms_min': 0.01,
        'median_decoded_torque_gain_min': 0.10,
        'positive_decoded_gain_fraction_min': 0.70,
    },
}
write_json_atomic(ANALYSIS_DIR / 'sanity_summary.json', sanity_summary)
display(pd.DataFrame({
    'diagnostic': [
        '17-point reconstruction slope',
        '17-point reconstruction Spearman',
        '17-point median normalized error',
        '9-vs-17 Spearman',
        '9-vs-17 median normalized difference',
        'fine-segment Spearman',
        'fine-segment median normalized error',
        'endpoint response RMS',
        'median decoded torque gain',
        'positive decoded gain fraction',
    ],
    'value': [
        reconstruction_slope, reconstruction_spearman,
        reconstruction_median_error, convergence_spearman,
        convergence_median_error, segment_spearman,
        segment_median_error, response_rms, median_decoded_gain,
        positive_decoded_gain_fraction,
    ],
}).round(4))
print('17-point reconstruction gate:', reconstruction_gate)
print('9-to-17 convergence gate:', convergence_gate)
print('Behavioral-response gate:', response_gate)
print('Probe-local-path gate:', probe_path_gate)
print('Path-integral estimator valid:', path_integral_estimator_valid)
print('Mediation interpretable:', mediation_interpretable)


In [ ]:
%%cell_progress 14/16 Estimate integrated mediation and its mechanism null
cell_stage('Computing path-integrated mediation coefficients')
rng = np.random.default_rng(SEED + SELECTED_LAYER_INDEX)
probe_response = path_frame['fine_probe_integral'].to_numpy(dtype=float)
final_response = path_frame[
    'fine_final_token_full_integral'
].to_numpy(dtype=float)
orth_response = path_frame[
    'fine_final_token_orthogonal_integral'
].to_numpy(dtype=float)
other_response = path_frame[
    'fine_other_token_integral'
].to_numpy(dtype=float)

def mediation_coefficient(response, target):
    return float(
        np.dot(response, target)
        / (np.dot(target, target) + np.finfo(float).eps)
    )

U_probe = mediation_coefficient(probe_response, observed)
U_final = mediation_coefficient(final_response, observed)
U_orth = mediation_coefficient(orth_response, observed)
U_other = mediation_coefficient(other_response, observed)
rho_probe = float(spearmanr(probe_response, observed).statistic)

cell_stage('Bootstrapping complete paths')
count = len(observed)
bootstrap_u = np.empty(BOOTSTRAP_REPLICATES, dtype=float)
bootstrap_rho = np.empty(BOOTSTRAP_REPLICATES, dtype=float)
for replicate in tqdm(
    range(BOOTSTRAP_REPLICATES), desc='Bootstrap path mediation', unit='rep'
):
    indices = rng.integers(0, count, size=count)
    bootstrap_u[replicate] = mediation_coefficient(
        probe_response[indices], observed[indices]
    )
    bootstrap_rho[replicate] = spearmanr(
        probe_response[indices], observed[indices]
    ).statistic
u_ci = [float(value) for value in np.quantile(bootstrap_u, [0.025, 0.975])]
rho_ci = [float(value) for value in np.nanquantile(
    bootstrap_rho, [0.025, 0.975]
)]

coordinate_matrix = np.asarray([
    [segment['probe_coordinate'] for segment in record['fine']['segments']]
    for record in records
], dtype=float)
sensitivity_matrix = np.asarray([
    [segment['probe_sensitivity'] for segment in record['fine']['segments']]
    for record in records
], dtype=float)
assert coordinate_matrix.shape == (
    PILOT_SCENES, FINE_PATH_POINTS - 1
)
coordinate_magnitude = np.linalg.norm(coordinate_matrix, axis=1)

# Preserve side and coarse natural tangent magnitude while breaking
# scene-specific alignment between the tangent and answer sensitivity.
strata = np.empty(count, dtype=int)
next_stratum = 0
sides = path_frame['target_side'].to_numpy()
for side in sorted(set(sides)):
    indices = np.flatnonzero(sides == side)
    order = indices[np.argsort(coordinate_magnitude[indices])]
    bin_count = min(2, max(1, len(order) // 3))
    for rank, index in enumerate(order):
        local_bin = min(bin_count - 1, rank * bin_count // len(order))
        strata[index] = next_stratum + local_bin
    next_stratum += bin_count
stratum_indices = [
    np.flatnonzero(strata == value) for value in sorted(np.unique(strata))
]
assert min(len(indices) for indices in stratum_indices) >= 2

cell_stage('Permuting whole probe-coordinate paths within matched strata')
permutation_u = np.empty(PERMUTATION_REPLICATES, dtype=float)
permutation_rho = np.empty(PERMUTATION_REPLICATES, dtype=float)
for replicate in tqdm(
    range(PERMUTATION_REPLICATES), desc='Integrated mechanism null', unit='rep'
):
    shuffled = coordinate_matrix.copy()
    for indices in stratum_indices:
        shuffled[indices] = shuffled[rng.permutation(indices)]
    shuffled_response = np.sum(shuffled * sensitivity_matrix, axis=1)
    permutation_u[replicate] = mediation_coefficient(
        shuffled_response, observed
    )
    permutation_rho[replicate] = spearmanr(
        shuffled_response, observed
    ).statistic
permutation_p_positive = float(
    (1 + np.sum(permutation_u >= U_probe))
    / (PERMUTATION_REPLICATES + 1)
)
permutation_p_two_sided = float(
    (1 + np.sum(np.abs(permutation_u) >= abs(U_probe)))
    / (PERMUTATION_REPLICATES + 1)
)
permutation_quantiles = [float(value) for value in np.quantile(
    permutation_u, [0.025, 0.50, 0.975]
)]

probe_mediation_supported = bool(
    mediation_interpretable
    and U_probe > 0 and u_ci[0] > 0 and rho_probe > 0
    and permutation_p_positive <= 0.05
)
if not path_integral_estimator_valid:
    status = 'path_integral_estimator_not_validated_do_not_interpret_U'
elif not probe_path_gate:
    status = 'path_integral_valid_global_probe_not_valid_on_local_paths'
elif probe_mediation_supported:
    status = 'integrated_selected_site_torque_mediation_supported_in_pilot'
else:
    status = 'integrated_selected_site_torque_mediation_not_supported_in_pilot'

mediation_summary = {
    'schema_version': 'path-integrated-mediation-summary-2.0',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'selected_layer_index': SELECTED_LAYER_INDEX,
    'n_paths': len(path_frame),
    'sanity_gates': sanity_summary,
    'primary': {
        'U_probe_integral': U_probe,
        'U_probe_bootstrap_ci95': u_ci,
        'probe_integral_spearman_with_observed_change': rho_probe,
        'probe_spearman_bootstrap_ci95': rho_ci,
        'U_final_token_all_directions': U_final,
        'U_final_token_probe_orthogonal': U_orth,
        'U_other_tokens': U_other,
    },
    'mechanism_null': {
        'strata_count': len(stratum_indices),
        'strata_sizes': [len(indices) for indices in stratum_indices],
        'U_null_quantiles_025_50_975': permutation_quantiles,
        'p_positive_add_one': permutation_p_positive,
        'p_two_sided_add_one': permutation_p_two_sided,
    },
    'probe_mediation_supported': probe_mediation_supported,
    'status': status,
    'interpretation_limit': (
        'This pilot tests one frozen layer and the final-token component of a '
        'global scene-level probe. U is an unbounded regression coefficient, '
        'not a literal percentage and not evidence of a unique circuit.'
    ),
}
write_json_atomic(ANALYSIS_DIR / 'mediation_summary.json', mediation_summary)
np.savez_compressed(
    ANALYSIS_DIR / 'resampling_nulls.npz',
    bootstrap_u=bootstrap_u, bootstrap_rho=bootstrap_rho,
    permutation_u=permutation_u, permutation_rho=permutation_rho,
    strata=strata,
)
display(pd.DataFrame({
    'quantity': ['U probe', 'U final all', 'U final orthogonal', 'U other tokens'],
    'estimate': [U_probe, U_final, U_orth, U_other],
}).round(5))
print('U probe CI95:', [round(value, 5) for value in u_ci])
print('Permutation p (positive):', round(permutation_p_positive, 4))
print('Status:', status)


In [ ]:
%%cell_progress 15/16 Plot reconstruction, convergence, and integrated mediation
cell_stage('Creating path-integral figures')
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
figure, axes = plt.subplots(2, 2, figsize=(12, 10))

axis = axes[0, 0]
axis.scatter(observed, fine_full, c=path_frame['center_signed_torque'], cmap='coolwarm', s=65)
bounds = np.array([
    min(observed.min(), fine_full.min()),
    max(observed.max(), fine_full.max()),
])
axis.plot(bounds, bounds, '--', color='black', linewidth=1)
axis.set_xlabel('Observed endpoint margin change')
axis.set_ylabel('17-point all-token integral')
axis.set_title(
    f'Reconstruction | rho={reconstruction_spearman:.2f}, '
    f'slope={reconstruction_slope:.2f}'
)

axis = axes[0, 1]
axis.scatter(fine_full, coarse_full, c=convergence_error, cmap='viridis', s=65)
bounds = np.array([
    min(fine_full.min(), coarse_full.min()),
    max(fine_full.max(), coarse_full.max()),
])
axis.plot(bounds, bounds, '--', color='black', linewidth=1)
axis.set_xlabel('17-point integral')
axis.set_ylabel('9-point integral')
axis.set_title(f'Numerical convergence | rho={convergence_spearman:.2f}')

axis = axes[1, 0]
colors = path_frame['target_side'].map({'left': '#2f6f9f', 'right': '#d17a22'})
axis.scatter(observed, probe_response, c=colors, s=65)
axis.axhline(0, color='black', linewidth=0.8)
axis.axvline(0, color='black', linewidth=0.8)
axis.set_xlabel('Observed endpoint margin change')
axis.set_ylabel('Integrated probe response')
axis.set_title(f'Probe-aligned natural path | U={U_probe:.4f}')

axis = axes[1, 1]
labels = ['probe', 'final orthogonal', 'other tokens']
values = [U_probe, U_orth, U_other]
axis.bar(labels, values, color=['#2f6f9f', '#9a8f7a', '#d17a22'])
axis.axhline(0, color='black', linewidth=0.8)
axis.set_ylabel('Regression-style path contribution U')
axis.set_title('Integrated response decomposition')
axis.tick_params(axis='x', rotation=15)

figure.suptitle(f'{MODEL_ID} — Path-Integrated TorqueTangent v2', fontsize=14)
figure.tight_layout()
figure_path = ANALYSIS_DIR / 'path_integrated_torque_tangent.png'
figure.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', figure_path)


In [ ]:
%%cell_progress 16/16 Export the estimator decision report
cell_stage('Writing the estimator decision report')
if not reconstruction_gate:
    recommendation = (
        'Do not interpret U. Increase path resolution from 17 to 33 points or '
        'reduce the ±4-pixel range in a new versioned estimator pilot.'
    )
elif not convergence_gate:
    recommendation = (
        'The line integral has not numerically converged. Run a 33-point grid '
        'before changing models or interpreting mediation.'
    )
elif not response_gate:
    recommendation = (
        'The local path produces too little behavioral change. Expand the pixel '
        'range slightly while retaining the path-integral convergence test.'
    )
elif not probe_path_gate:
    recommendation = (
        'The estimator is usable, but the global scene-level probe does not track '
        'local physical motion. Fit a scene-disjoint path-tangent probe on '
        'train/dev paths, then evaluate it on these untouched probe-test paths.'
    )
elif probe_mediation_supported:
    recommendation = (
        'Freeze this estimator and run a 40-scene Qwen confirmatory pass. Only '
        'after that result should the identical protocol be run on Gemma.'
    )
else:
    recommendation = (
        'The estimator is valid, but the frozen final-token torque direction does '
        'not mediate the natural response. Next localize the integrated response '
        'across token groups at this layer before examining neighboring layers.'
    )

decision_report = {
    'schema_version': 'path-integral-decision-2.0',
    'model_id': MODEL_ID,
    'status': status,
    'recommendation': recommendation,
    'artifacts': {
        'protocol': str(PROTOCOL_LOCK_PATH),
        'capture_config': str(CAPTURE_CONFIG_PATH),
        'path_manifest': str(path_manifest_path),
        'primary_csv': str(ANALYSIS_DIR / 'path_integral_primary.csv'),
        'segment_csv': str(ANALYSIS_DIR / 'path_integral_segments.csv'),
        'sanity_summary': str(ANALYSIS_DIR / 'sanity_summary.json'),
        'mediation_summary': str(ANALYSIS_DIR / 'mediation_summary.json'),
        'resampling_nulls': str(ANALYSIS_DIR / 'resampling_nulls.npz'),
        'figure': str(ANALYSIS_DIR / 'path_integrated_torque_tangent.png'),
    },
}
write_json_atomic(ANALYSIS_DIR / 'decision_report.json', decision_report)
print('Decision:', status)
print('Next:', recommendation)
print('Do not run Gemma unless mediation_interpretable is True.')
